# PyYAML examples

## Example 1: Load and dump YAML

In [ ]:
import yaml

print("Example 1: Basic YAML loading and dumping operations")

# Sample YAML data as string
yaml_data = """
name: John Doe
age: 30
occupation: Software Engineer
contacts:
  email: john.doe@example.com
  phone: "+1-555-0123"
skills:
  - Python
  - JavaScript  
  - Docker
  - Kubernetes
active: true
salary: 75000.50
"""

print("Original YAML string:")
print(yaml_data)

# Load YAML data into Python object
data = yaml.safe_load(yaml_data)
print(f"\nLoaded Python data type: {type(data)}")
print(f"Name: {data['name']}")
print(f"Age: {data['age']}")
print(f"Skills: {data['skills']}")
print(f"Email: {data['contacts']['email']}")

# Dump Python object back to YAML
dumped_yaml = yaml.dump(data, default_flow_style=False)
print(f"\nDumped back to YAML:")
print(dumped_yaml)

# Verify round-trip consistency
reloaded_data = yaml.safe_load(dumped_yaml)
print(f"Round-trip successful: {data == reloaded_data}")
assert data == reloaded_data, (data, reloaded_data)

print("Example 1 completed successfully!")

## Example 2: Safe loading and parse errors

In [ ]:
import yaml

print("Example 2: Safe loading and security considerations")

# Safe YAML data - basic data types only
safe_yaml = """
user_config:
  username: "john_doe"
  preferences:
    theme: "dark"
    notifications: true
    timeout: 300
  allowed_actions:
    - read
    - write
    - delete
"""

# Plain data: safe_load builds only dicts, lists, strings and numbers from it
potentially_unsafe_yaml = """
data:
  basic_info: "safe string"
  number: 42
  
# Note: In real scenarios, unsafe YAML might contain 
# Python object constructors that could execute code
"""

print("=== Safe Loading (Recommended) ===")
# Always use safe_load for untrusted input
safe_data = yaml.safe_load(safe_yaml)
print(f"Safe data loaded: {type(safe_data)}")
print(f"Username: {safe_data['user_config']['username']}")
print(f"Theme: {safe_data['user_config']['preferences']['theme']}")
print(f"Actions: {', '.join(safe_data['user_config']['allowed_actions'])}")

# Demonstrate safe loading with potentially unsafe content
print("\n=== Safe Loading with Potentially Unsafe Content ===")
safe_unsafe_data = yaml.safe_load(potentially_unsafe_yaml)
print(f"Safely loaded data: {safe_unsafe_data}")

# Error handling for invalid YAML
print("\n=== Error Handling ===")
invalid_yaml = """
key1: value1
key2: value2
  invalid_indentation: value3  # This will cause an error
key3: value3
"""

try:
    yaml.safe_load(invalid_yaml)
except yaml.YAMLError as e:
    print(f"✓ YAML parsing error caught: {type(e).__name__}")
    print(f"  Error details: {str(e)[:100]}")
    assert e.problem_mark.line == 3, e.problem_mark  # 0-based: the badly indented fourth line
else:
    raise AssertionError("safe_load accepted badly indented YAML")

# safe_load refuses tags that would construct arbitrary Python objects
try:
    yaml.safe_load("!!python/object/apply:os.getcwd []")
except yaml.constructor.ConstructorError as e:
    print(f"✓ Python object tag rejected: {e.problem}")
else:
    raise AssertionError("safe_load constructed a Python object from a !!python tag")

assert safe_data["user_config"]["preferences"] == {"theme": "dark", "notifications": True, "timeout": 300}
assert safe_unsafe_data == {"data": {"basic_info": "safe string", "number": 42}}, safe_unsafe_data

# Best practices summary
print("\n=== Security Best Practices ===")
print("✓ Always use yaml.safe_load() for untrusted input")
print("✓ Never use yaml.load() without specifying a Loader")
print("✓ Validate loaded data structure before use")
print("✓ Handle YAML parsing errors gracefully")
print("✓ Consider size limits for input YAML files")

print("Example 2 completed successfully!")

## Example 3: Configuration files and complex data types

In [ ]:
import yaml
from datetime import date, datetime
from decimal import Decimal

print("Example 3: Configuration files and complex data types")

# Application configuration in YAML format (all values are placeholders)
app_config_yaml = """
# Application Configuration
app:
  name: "MyWebApp"
  version: "2.1.0"
  debug: false

database:
  host: "localhost"
  port: 5432
  name: "myapp"
  credentials:
    username: "app_user"
    password: "change-me"
  pool:
    min_connections: 5
    max_connections: 50
    timeout: 30

logging:
  level: "INFO"
  format: "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
  handlers:
    - console
    - file
  file_config:
    filename: "app.log"
    max_bytes: 10485760  # 10MB
    backup_count: 5

features:
  user_registration: true
  email_verification: true
  two_factor_auth: false
  analytics: true

cache:
  type: "redis"
  settings:
    host: "127.0.0.1"
    port: 6379
    db: 0
    expire: 3600
"""

config = yaml.safe_load(app_config_yaml)

print(f"Application: {config['app']['name']} v{config['app']['version']}")
print(f"Database: {config['database']['host']}:{config['database']['port']}")
print(f"Cache type: {config['cache']['type']}")
print(f"Features enabled: {sum(config['features'].values())} of {len(config['features'])}")
assert config["app"] == {"name": "MyWebApp", "version": "2.1.0", "debug": False}, config["app"]
assert config["database"]["port"] == 5432 and config["database"]["pool"]["max_connections"] == 50
assert config["logging"]["file_config"]["max_bytes"] == 10 * 1024 * 1024  # the trailing comment is ignored
assert sum(config["features"].values()) == 3

# Dates and datetimes round-trip through the safe dumper and loader
dated = {"created_at": datetime(2024, 1, 15, 9, 30), "release_date": date(2024, 1, 15)}
dated_yaml = yaml.safe_dump(dated)
print("\nDates as YAML:")
print(dated_yaml)
assert yaml.safe_load(dated_yaml) == dated

# A Decimal has no plain YAML form: safe_dump refuses it, and yaml.dump writes a Python-specific tag
complex_data = {"price": Decimal("99.99"), "tags": ["production", "stable", "v2"]}
try:
    yaml.safe_dump(complex_data)
except yaml.representer.RepresenterError as e:
    print(f"✓ safe_dump refuses Decimal: {e.args[0]}")
else:
    raise AssertionError("safe_dump wrote a Decimal")

complex_yaml = yaml.dump(complex_data, default_flow_style=False)
print("yaml.dump output:")
print(complex_yaml)
assert "!!python/object/apply:decimal.Decimal" in complex_yaml, complex_yaml
try:
    yaml.safe_load(complex_yaml)
except yaml.constructor.ConstructorError:
    print("✓ safe_load will not read the Python-specific tag back")
else:
    raise AssertionError("safe_load constructed a Decimal from a !!python tag")

print("Example 3 completed successfully!")

## Example 4: Custom tags, multiple documents and block-style strings

In [ ]:
import os
import re

import yaml

print("Example 4: Custom tags, multiple documents and block-style strings")


class ConfigLoader(yaml.SafeLoader):
    """A SafeLoader with two extra tags. A subclass keeps yaml.safe_load itself unchanged."""


def env_var_constructor(loader, node):
    """!env 'NAME|default' reads an environment variable, with an optional default."""
    value = loader.construct_scalar(node)
    name, _, default = value.partition("|")
    return os.environ.get(name.strip(), default.strip())


def regex_constructor(loader, node):
    """!regex 'pattern' compiles a regular expression."""
    return re.compile(loader.construct_scalar(node))


ConfigLoader.add_constructor("!env", env_var_constructor)
ConfigLoader.add_constructor("!regex", regex_constructor)

custom_yaml = """
app_settings:
  database_url: !env 'PYYAML_DEMO_DATABASE_URL|postgresql://localhost:5432/myapp'
  debug_mode: !env 'PYYAML_DEMO_DEBUG|false'

validation:
  email_pattern: !regex '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'
  phone_pattern: !regex '^\\+?1?\\d{9,15}$'
"""

print("=== Loading YAML with Custom Constructors ===")
os.environ.pop("PYYAML_DEMO_DATABASE_URL", None)  # unset: the default from the tag is used
os.environ["PYYAML_DEMO_DEBUG"] = "true"  # set: the environment wins
try:
    config = yaml.load(custom_yaml, Loader=ConfigLoader)
finally:
    os.environ.pop("PYYAML_DEMO_DEBUG", None)

print(f"Database URL: {config['app_settings']['database_url']}")
print(f"Debug mode: {config['app_settings']['debug_mode']}")
assert config["app_settings"] == {"database_url": "postgresql://localhost:5432/myapp", "debug_mode": "true"}

email_pattern = config["validation"]["email_pattern"]
phone_pattern = config["validation"]["phone_pattern"]
assert isinstance(email_pattern, re.Pattern)
assert email_pattern.match("user@example.com") and not email_pattern.match("not-an-email")
assert phone_pattern.match("+1234567890") and not phone_pattern.match("12-34")
print("Email and phone patterns compiled and matched")

# The plain safe loader does not know the custom tags
try:
    yaml.safe_load(custom_yaml)
except yaml.constructor.ConstructorError as e:
    print(f"✓ yaml.safe_load rejects the custom tag: {e.problem}")
else:
    raise AssertionError("yaml.safe_load accepted the !env tag")

print("\n=== Streaming Multiple YAML Documents ===")
multi_doc_yaml = """
---
document: 1
type: user
data:
  name: "Alice"
  role: "admin"
---
document: 2
type: user
data:
  name: "Bob"
  role: "user"
---
document: 3
type: config
data:
  timeout: 300
  retries: 3
"""

documents = list(yaml.safe_load_all(multi_doc_yaml))
print(f"Loaded {len(documents)} documents:")
for i, doc in enumerate(documents, 1):
    print(f"  Document {i}: {doc['type']} - {doc.get('data', {})}")
assert [doc["document"] for doc in documents] == [1, 2, 3]
assert [doc["type"] for doc in documents] == ["user", "user", "config"]
assert list(yaml.safe_load_all(yaml.safe_dump_all(documents))) == documents

print("\n=== Custom Representer ===")


class BlockStyleDumper(yaml.SafeDumper):
    """A SafeDumper that writes multi-line strings as | blocks."""


def str_presenter(dumper, data):
    style = "|" if len(data.splitlines()) > 1 else None
    return dumper.represent_scalar("tag:yaml.org,2002:str", data, style=style)


BlockStyleDumper.add_representer(str, str_presenter)

multiline_data = {
    "description": "This is a long description\nthat spans multiple lines\nand should be formatted nicely",
    "short": "single line",
}
yaml_output = yaml.dump(multiline_data, Dumper=BlockStyleDumper, default_flow_style=False)
print("Multiline string handling:")
print(yaml_output)
assert yaml_output.startswith("description: |-\n"), yaml_output
assert yaml.safe_load(yaml_output) == multiline_data
assert "|" not in yaml.safe_dump(multiline_data)  # the default dumper is unchanged

print("Example 4 completed successfully!")